# 03 - Testando o contrato da API

Aqui usamos o `TestClient` do FastAPI. Não é preciso iniciar o Uvicorn. Os erros são entradas inválidas controladas: cada cenário declara o status esperado e confere a resposta.

Isso verifica o contrato e a integração com o pipeline. Não mede desempenho de rede nem a qualidade preditiva do modelo.

In [1]:
from pathlib import Path
import sys


print("Python usado pelo notebook:")
print(sys.executable)


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/churn").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import copy
import json
import tempfile
import pandas as pd
from fastapi.testclient import TestClient
from IPython.display import display
from churn.api import create_app
from churn.config import MODEL_PATH

payload = json.loads((ROOT / "docs/example_customer.json").read_text())
assert MODEL_PATH.exists(), "Execute python -m churn.train antes deste notebook."


Python usado pelo notebook:
c:\Users\Marco-pc\Documents\github\churn-tech-challenge-revisado\churn-tech-challenge_rewisado\.venv\Scripts\python.exe


c:\Users\Marco-pc\Documents\github\churn-tech-challenge-revisado\churn-tech-challenge_rewisado\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Cliente válido e campos problemáticos

Uma coluna obrigatória ausente não é igual a uma cobrança total ausente: `tenure` é obrigatório, enquanto `TotalCharges` pode ser imputado. Campos extras também são rejeitados para detectar erros de integração.

In [11]:
def changed(**fields):
    customer = copy.deepcopy(payload)
    customer.update(fields)
    return customer

without_tenure = copy.deepcopy(payload)
without_tenure.pop("tenure")
without_total = copy.deepcopy(payload)
without_total.pop("TotalCharges")
scenarios = [
    ("Cliente válido", copy.deepcopy(payload), 200),
    ("TotalCharges omitido", without_total, 200),
    ("tenure omitido", without_tenure, 422),
    ("tenure negativo", changed(tenure=-1), 422),
    ("Contrato desconhecido", changed(Contract="Mensal"), 422),
    ("Alvo enviado por engano", changed(Churn="Yes"), 422),
    ("Serviços incompatíveis", changed(PhoneService="No", MultipleLines="Yes"), 422),
]
results = []
with TestClient(create_app()) as client:
    health = client.get("/health")
    assert health.status_code == 200 and health.json()["model_loaded"]
    for name, customer, expected in scenarios:
        response = client.post("/predict", json=customer)
        assert response.status_code == expected, (name, response.text)
        body = response.json()
        if expected == 200:
            assert 0 <= body["churn_probability"] <= 1
            assert body["churn_prediction"] == int(body["churn_probability"] >= body["threshold"])
        results.append({"scenario": name, "expected_status": expected,
                        "actual_status": response.status_code, "response": body})
display(pd.DataFrame(results)[["scenario", "expected_status", "actual_status"]])
# Mostrar um erro ajuda a entender onde o contrato rejeitou o campo.
display(results[3]["response"])

,scenario,expected_status,actual_status
0,Cliente válido,200,200
1,TotalCharges omitido,200,200
2,tenure omitido,422,422
3,tenure negativo,422,422
4,Contrato desconhecido,422,422
5,Alvo enviado por engano,422,422
6,Serviços incompatíveis,422,422


{'detail': [{'type': 'greater_than_equal',
   'loc': ['body', 'tenure'],
   'msg': 'Input should be greater than or equal to 0',
   'input': -1,
   'ctx': {'ge': 0}}]}

## API ativa, mas sem modelo

`/health` é uma verificação de processo: pode retornar 200 mesmo sem modelo. Para saber se há inferência disponível, precisamos olhar `model_loaded`. O `/predict` responde 503 nessa situação.

In [12]:
with tempfile.TemporaryDirectory() as directory:
    with TestClient(create_app(Path(directory) / "missing.joblib")) as client:
        health = client.get("/health")
        prediction = client.post("/predict", json=payload)
        assert health.status_code == 200
        assert health.json()["model_loaded"] is False
        assert prediction.status_code == 503
        print("Health:", health.json())
        print("Predict:", prediction.status_code, prediction.json())
        results.append({"scenario": "Modelo ausente", "expected_status": 503,
                        "actual_status": prediction.status_code, "response": prediction.json()})
OUT = ROOT / "reports/review"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "api_scenarios.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(results)} cenários conferidos.")

Health: {'status': 'ok', 'model_loaded': False}
Predict: 503 {'detail': 'Modelo indisponível. Execute python -m churn.train e reinicie a API.'}
8 cenários conferidos.


## Observações da revisão

- Status 422 indica que o cliente precisa corrigir a entrada; repetir o mesmo JSON não resolve.
- Status 503 indica indisponibilidade do modelo. É preciso treinar ou configurar o artefato e reiniciar a API.
- O notebook não inicia servidor TCP e não comprova abertura do Swagger no Windows. Essa conferência continua sendo local.
- O exemplo de cliente não tem identificador. Não precisamos enviar dados pessoais para demonstrar a integração.

In [13]:
import sys
import subprocess

ROOT = next(
    pasta
    for pasta in [Path.cwd(), *Path.cwd().parents]
    if (pasta / "src" / "churn").is_dir()
)

print("Python usado:", sys.executable)
print("Pasta do projeto:", ROOT)

# Instala as dependências no mesmo Python que executa o notebook.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    str(ROOT),
])

Python usado: c:\Users\Marco-pc\Documents\github\churn-tech-challenge-revisado\churn-tech-challenge_rewisado\.venv\Scripts\python.exe
Pasta do projeto: c:\Users\Marco-pc\Documents\github\churn-tech-challenge-revisado\churn-tech-challenge_rewisado


0